# 02 — Build JSONL datasets with PhysioNet Access


In [ ]:
!pip -q install pyyaml jsonschema pydantic pandas
import os
from pathlib import Path
import sys; sys.path.append('/kaggle/working/criticalcare-copilot')


### Download PhysioNet Core Tables
We use the provided credentials to fetch only the metadata tables needed for case normalization. This avoids downloading terabytes of waveform/chart data.

In [ ]:
user = 'hssling@yahoo.com'
password = 'Sidda@1000'

# MIMIC-IV Core tables
!mkdir -p /kaggle/working/data/raw/mimiciv/hosp /kaggle/working/data/raw/mimiciv/icu
mimic_files = [
    'hosp/patients.csv.gz', 'hosp/admissions.csv.gz',
    'icu/icustays.csv.gz'
]
for f in mimic_files:
    url = f'https://physionet.org/content/mimiciv/2.2/{f}'
    dest = f'/kaggle/working/data/raw/mimiciv/{f}'
    !wget --user {user} --password {password} -O {dest} {url}

# eICU Core tables
!mkdir -p /kaggle/working/data/raw/eicu
eicu_files = ['patient.csv.gz']
for f in eicu_files:
    url = f'https://physionet.org/content/eicu-crd/2.0/{f}'
    dest = f'/kaggle/working/data/raw/eicu/{f}'
    !wget --user {user} --password {password} -O {dest} {url}

os.environ['MIMIC_IV_ROOT'] = '/kaggle/working/data/raw/mimiciv'
os.environ['EICU_ROOT'] = '/kaggle/working/data/raw/eicu'


In [ ]:
!python -m data.builders.build_mimic_cases --out /kaggle/working/processed/mimic_cases.jsonl --limit 2000
!python -m data.builders.build_eicu_cases   --out /kaggle/working/processed/eicu_cases.jsonl  --limit 2000
!python -m data.builders.build_n2c2_ade_cases --out /kaggle/working/processed/ade_cases.jsonl
!python -m data.builders.build_safety_pairs --out /kaggle/working/processed/safety_pairs.jsonl
!python -m data.builders.split_train_valid_test --inputs /kaggle/working/processed/*.jsonl --out /kaggle/working/processed
